Minimal CPU smoke test. Verifies shapes thread end-to-end and gradients flow with B=2, G=2, max_completion_length=16 before running on GPU. Completes in <1 minute on CPU

In [ ]:
%load_ext autoreload                                                                                                                              
%autoreload 2 
import sys
from pathlib import Path
sys.path.insert(0,str(Path.cwd().parent))

In [ ]:
import yaml
from src.grpo import GRPOTrainer
from transformers import AutoTokenizer
from src.data import load_gsm8k
from src.rewards import binary_reward
from torch.utils.data import DataLoader

In [ ]:
config = yaml.safe_load(Path("../configs/smoke.yaml").read_text())
model_name = config["model"]
tokenizer = AutoTokenizer.from_pretrained(model_name)
dataset = load_gsm8k(tokenizer, split="train[:8]")
dataset = dataset.select_columns(["prompt","answer"])
loader = DataLoader(dataset, batch_size=config["batch_size"], shuffle=False)
trainer = GRPOTrainer(model_name, tokenizer, binary_reward, config, "cpu")

trainer.train(loader)